In [1]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

Num GPUs Available:  1


In [2]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Input, Conv2D, BatchNormalization, ReLU, Add, Concatenate
from tensorflow.keras import Model

# Function to convert DCT coefficients to binary volume representation
#T from 20 to 10
def dct_volume_representation(dct_coeffs, T=10):
    # Clip the DCT coefficients
    clipped_coeffs = tf.clip_by_value(dct_coeffs, -T, T)
    abs_coeffs = tf.abs(clipped_coeffs)

    # Create the binary volume
    binary_volume = tf.cast(abs_coeffs > 0, tf.float32)
    return binary_volume

from tensorflow.keras.layers import concatenate

#def frequency_separation(inputs):
#    low_freq = Conv2D(256, (1, 1), padding='same', activation='relu')(inputs)
#    high_freq = Conv2D(256, (3, 3), padding='same', activation='relu')(inputs)
#    combined = concatenate([low_freq, high_freq], axis=-1)
#    return combined

def dilated_conv_bn_relu(inputs, filters, kernel_size=(3, 3), dilation_rate=(2, 2), padding='same'):
    x = Conv2D(filters, kernel_size, dilation_rate=dilation_rate, padding=padding)(inputs)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    return x


def conv_bn_relu(inputs, filters, kernel_size=(3, 3), strides=(1, 1), padding='same'):
    x = Conv2D(filters, kernel_size, strides=strides, padding=padding)(inputs)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    return x

def grid_aligned_cropping(inputs, crop_height, crop_width, i, j):
    if crop_height % 8 != 0 or crop_width % 8 != 0 or i % 8 != 0 or j % 8 != 0:
        raise ValueError("crop_height, crop_width, i, and j must be multiples of 8")
    
    cropped = inputs[:, i:i + crop_height, j:j + crop_width, :]
    
    return cropped

In [3]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Concatenate, Input
from tensorflow.keras.models import Model

def repeat_quantization_table(quant_table, target_shape):
    quant_table = tf.constant(quant_table, dtype=tf.float32)
    quant_table = tf.expand_dims(quant_table, axis=-1)
    repeated_table = tf.tile(quant_table, [target_shape[0], target_shape[1], 1])
    return repeated_table

import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D, BatchNormalization, ReLU, concatenate

class JPEGALMLayer(Layer):
    def __init__(self, quant_table, T, filters_dct, filters_quant, **kwargs):
        super(JPEGALMLayer, self).__init__(**kwargs)
        self.quant_table = quant_table
        self.T = T
        self.filters_dct = filters_dct
        self.filters_quant = filters_quant

    def build(self, input_shape):
        self.dilated_conv_bn_relu = tf.keras.Sequential([
            Conv2D(filters=self.filters_dct, kernel_size=3, padding='same', dilation_rate=2),
            BatchNormalization(),
            ReLU()
        ])

        self.conv_bn_relu = tf.keras.Sequential([
            Conv2D(filters=self.filters_dct, kernel_size=3, padding='same'),  # Match filters_dct
            BatchNormalization(),
            ReLU()
        ])

        self.conv_low_freq = Conv2D(128, (1, 1), padding='same', activation='relu')
        self.conv_high_freq = Conv2D(128, (3, 3), padding='same', activation='relu')

    def call(self, inputs):
        # Perform DCT on inputs
        dct_coeffs = tf.signal.dct(inputs, type=2, norm='ortho')
        
        # Quantization based on quantization table
        repeated_table = tf.constant(self.quant_table, dtype=tf.float32)
        repeated_table = tf.expand_dims(repeated_table, axis=0)
        repeated_table = tf.tile(repeated_table, [tf.shape(dct_coeffs)[0], 1, 1])
        repeated_table = tf.expand_dims(repeated_table, axis=-1)

        x_dct = self.dilated_conv_bn_relu(dct_coeffs)
        x_quant = self.conv_bn_relu(repeated_table)

        # Ensure x_dct and x_quant have compatible shapes
        x_quant = tf.image.resize(x_quant, tf.shape(x_dct)[1:3])  # Resize x_quant to match x_dct's spatial dimensions

        x_combined = x_dct * x_quant
        x_combined_separated = self.frequency_separation(x_combined)
        
        return x_combined_separated

    def get_config(self):
        config = super(JPEGALMLayer, self).get_config()
        config.update({
            'quant_table': self.quant_table,
            'T': self.T,
            'filters_dct': self.filters_dct,
            'filters_quant': self.filters_quant
        })
        return config
    
    def frequency_separation(self, inputs):
        low_freq = self.conv_low_freq(inputs)
        high_freq = self.conv_high_freq(inputs)
        combined = concatenate([low_freq, high_freq], axis=-1)
        return combined
        
    @classmethod
    def from_config(cls, config):
        return cls(**config)

In [4]:
def convolutional_unit(input_tensor, filters1, filters2, num_repeats=4):

    x = input_tensor
    #print(x.shape)
    
    for _ in range(num_repeats):
        # Initial branch to set aside
        initial_branch = tf.keras.layers.Conv2D(filters2, kernel_size=1, padding='same')(x)
        
        # Other branch
        x = tf.keras.layers.Conv2D(filters=filters1, kernel_size=3, strides=1, padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)

        x = tf.keras.layers.Conv2D(filters=filters2, kernel_size=3, strides=1, padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)

        # Element-wise addition with the initial branch
        x = tf.keras.layers.Add()([x, initial_branch])

        # Final ReLU activation
        x = tf.keras.layers.ReLU()(x)

    return x

import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D

class UpSample1x1Bilinear(Layer):
    def __init__(self, scale_factor=2, filters=None, **kwargs):
        super().__init__(**kwargs)
        self.scale_factor = scale_factor
        self.filters = filters
        self.conv = None

    def build(self, input_shape):
        # If filters are not provided, use the same number of input channels
        if self.filters is None:
            self.filters = input_shape[-1]
        # Define the 1x1 convolution layer
        self.conv = Conv2D(filters=self.filters, kernel_size=1, padding='same')

    def call(self, input_tensor):
        # Apply bilinear upsampling
        upsampled = tf.image.resize(input_tensor, 
                                    size=[tf.shape(input_tensor)[1] * self.scale_factor, 
                                          tf.shape(input_tensor)[2] * self.scale_factor], 
                                    method='bilinear')
        # Apply 1x1 convolution
        output = self.conv(upsampled)
        return output

    def get_config(self):
        config = super().get_config()
        config.update({
            "scale_factor": self.scale_factor,
            "filters": self.filters,
        })
        return config


def downsample_stride2_conv(input_tensor, filters):

    output_tensor = tf.keras.layers.Conv2D(filters=filters, kernel_size=3, strides=2, padding='same')(input_tensor)
    return output_tensor
    

In [5]:
def fusion_block_3inputs(input1, input2, input3):
    channels1 = input1.shape[-1]
    channels2 = input2.shape[-1]
    channels3 = input3.shape[-1]

    #print(input1.shape)
    #print(input2.shape)
    #print(input3.shape)
    # Adjust number of channels in input2 and input3 to match input1
    #input2_adjusted = Conv2D(channels1, kernel_size=1, padding='same')(input2)
    #input3_adjusted = Conv2D(channels1, kernel_size=1, padding='same')(input3)

    # Upsample input2 and input3 to match input1
    upsampled_input2 = UpSample1x1Bilinear(filters=channels1)(input2)
    upsampled_input3_twice = UpSample1x1Bilinear(filters=channels1)(UpSample1x1Bilinear(filters=channels2)(input3))
    print(input1.shape)
    print(upsampled_input2.shape)
    print(upsampled_input3_twice.shape)

    # Largest output feature map
    output1 = Add()([input1, upsampled_input2, upsampled_input3_twice])

    # Adjust number of channels in downsampled input1 and upsampled input3 to match input2
    downsampled_input1 = downsample_stride2_conv(input1, filters=channels2)
    downsampled_input1_adjusted = Conv2D(channels2, kernel_size=1, padding='same')(downsampled_input1)
    upsampled_input3_adjusted = Conv2D(channels2, kernel_size=1, padding='same')(UpSample1x1Bilinear(filters=channels2)(input3))

    # Middle output feature map
    output2 = Add()([downsampled_input1_adjusted, input2, upsampled_input3_adjusted])

    # Adjust number of channels in downsampled input1 and input2 to match input3
    downsampled_input1_twice = downsample_stride2_conv(downsample_stride2_conv(input1, filters=channels3), filters=channels3)
    downsampled_input2 = downsample_stride2_conv(input2, filters=channels3)
    downsampled_input2_adjusted = Conv2D(channels3, kernel_size=1, padding='same')(downsampled_input2)

    # Smallest output feature map
    output3 = Add()([downsampled_input1_twice, downsampled_input2_adjusted, input3])

    return output1, output2, output3
    

In [6]:
def fusion_block_4inputs(input1, input2, input3, input4):
    channels1 = input1.shape[-1]
    channels2 = input2.shape[-1]
    channels3 = input3.shape[-1]
    channels4 = input4.shape[-1]

    #print(input1.shape)
    #print(input2.shape)
    #print(input3.shape)
    #print(input4.shape)
    # Adjust number of channels in input2, input3, and input4 to match input1
    #input2_adjusted = tf.keras.layers.Conv2D(channels1, kernel_size=1, padding='same')(input2)
    #input3_adjusted = tf.keras.layers.Conv2D(channels1, kernel_size=1, padding='same')(input3)
    #input4_adjusted = tf.keras.layers.Conv2D(channels1, kernel_size=1, padding='same')(input4)

    # Upsample input2, input3, and input4 to match input1
    upsampled_input2 = UpSample1x1Bilinear(filters=24)(input2)
    upsampled_input3_twice = UpSample1x1Bilinear(filters=24)(UpSample1x1Bilinear(filters=48)(input3))
    upsampled_input4_four_times = UpSample1x1Bilinear(filters=24)(UpSample1x1Bilinear(filters=48)(UpSample1x1Bilinear(filters=96)(input4)))
    
    # Largest output feature map
    output1 = tf.keras.layers.Add()([input1, upsampled_input2, upsampled_input3_twice, upsampled_input4_four_times])

    #print(input2.shape)
    #print(input3.shape)

    # Adjust number of channels in downsampled input1, upsampled input3, and upsampled input4 to match input2
    downsampled_input1 = downsample_stride2_conv(input1, filters=channels2)
    downsampled_input1_adjusted = tf.keras.layers.Conv2D(channels2, kernel_size=1, padding='valid')(downsampled_input1)
    upsampled_input3_adjusted = UpSample1x1Bilinear(filters=48)(input3)
    upsampled_input4_twice_adjusted = UpSample1x1Bilinear(filters=48)(UpSample1x1Bilinear(filters=96)(input4))

    print(downsampled_input1_adjusted.shape)
    print(input2.shape)
    print(upsampled_input3_adjusted.shape)
    print(upsampled_input4_twice_adjusted.shape)
    # Second largest output feature map
    output2 = tf.keras.layers.Add()([downsampled_input1_adjusted, input2, upsampled_input3_adjusted, upsampled_input4_twice_adjusted])

    # Adjust number of channels in downsampled input2, downsampled input1, and upsampled input4 to match input3
    downsampled_input2 = downsample_stride2_conv(input2, filters=channels3)
    downsampled_input1_twice = downsample_stride2_conv(downsample_stride2_conv(input1, filters=channels3), filters=channels3)
    downsampled_input1_twice_adjusted = tf.keras.layers.Conv2D(channels3, kernel_size=1, padding='same')(downsampled_input1_twice)
    upsampled_input4_adjusted = tf.keras.layers.Conv2D(channels3, kernel_size=1, padding='same')(UpSample1x1Bilinear(filters=96)(input4))

    # Second smallest output feature map
    output3 = tf.keras.layers.Add()([downsampled_input1_twice_adjusted, downsampled_input2, input3, upsampled_input4_adjusted])

    # Adjust number of channels in downsampled input3, downsampled input2, and downsampled input1 to match input4
    downsampled_input3 = downsample_stride2_conv(input3, filters=channels4)
    downsampled_input2_twice = downsample_stride2_conv(downsample_stride2_conv(input2, filters=channels4), filters=channels4)
    downsampled_input2_twice_adjusted = tf.keras.layers.Conv2D(channels4, kernel_size=1, padding='same')(downsampled_input2_twice)
    downsampled_input1_four_times = downsample_stride2_conv(downsample_stride2_conv(downsample_stride2_conv(input1, filters=channels4), filters=channels4), filters=channels4)
    downsampled_input1_four_times_adjusted = tf.keras.layers.Conv2D(channels4, kernel_size=1, padding='same')(downsampled_input1_four_times)

    # Smallest output feature map
    output4 = tf.keras.layers.Add()([downsampled_input1_four_times_adjusted, downsampled_input2_twice_adjusted, downsampled_input3, input4])

    return output1, output2, output3, output4

In [7]:
def fusion_block_4inputs_fusion_stream(input1, input2, input3, input4):
    channels1 = input1.shape[-1]
    channels2 = input2.shape[-1]
    channels3 = input3.shape[-1]
    channels4 = input4.shape[-1]

    #print(input1.shape)
    #print(input2.shape)
    #print(input3.shape)
    #print(input4.shape)
    # Adjust number of channels in input2, input3, and input4 to match input1
    #input2_adjusted = tf.keras.layers.Conv2D(channels1, kernel_size=1, padding='same')(input2)
    #input3_adjusted = tf.keras.layers.Conv2D(channels1, kernel_size=1, padding='same')(input3)
    #input4_adjusted = tf.keras.layers.Conv2D(channels1, kernel_size=1, padding='same')(input4)

    # Upsample input2, input3, and input4 to match input1
    upsampled_input2 = UpSample1x1Bilinear(filters=6)(input2)
    upsampled_input3_twice = UpSample1x1Bilinear(filters=6)(UpSample1x1Bilinear(filters=24)(input3))
    upsampled_input4_four_times = UpSample1x1Bilinear(filters=6)(UpSample1x1Bilinear(filters=24)(UpSample1x1Bilinear(filters=48)(input4)))
    
    # Largest output feature map
    output1 = tf.keras.layers.Add()([input1, upsampled_input2, upsampled_input3_twice, upsampled_input4_four_times])

    #print(input2.shape)
    #print(input3.shape)

    # Adjust number of channels in downsampled input1, upsampled input3, and upsampled input4 to match input2
    downsampled_input1 = downsample_stride2_conv(input1, filters=channels2)
    downsampled_input1_adjusted = tf.keras.layers.Conv2D(24, kernel_size=1, padding='valid')(downsampled_input1)
    upsampled_input3_adjusted = UpSample1x1Bilinear(filters=24)(input3)
    upsampled_input4_twice_adjusted = UpSample1x1Bilinear(filters=24)(UpSample1x1Bilinear(filters=48)(input4))

    print(downsampled_input1_adjusted.shape)
    print(input2.shape)
    print(upsampled_input3_adjusted.shape)
    print(upsampled_input4_twice_adjusted.shape)
    # Second largest output feature map
    output2 = tf.keras.layers.Add()([downsampled_input1_adjusted, input2, upsampled_input3_adjusted, upsampled_input4_twice_adjusted])

    # Adjust number of channels in downsampled input2, downsampled input1, and upsampled input4 to match input3
    downsampled_input2 = downsample_stride2_conv(input2, filters=channels3)
    downsampled_input1_twice = downsample_stride2_conv(downsample_stride2_conv(input1, filters=channels3), filters=96)
    downsampled_input1_twice_adjusted = tf.keras.layers.Conv2D(48, kernel_size=1, padding='same')(downsampled_input1_twice)
    upsampled_input4_adjusted = tf.keras.layers.Conv2D(48, kernel_size=1, padding='same')(UpSample1x1Bilinear(filters=48)(input4))

    # Second smallest output feature map
    output3 = tf.keras.layers.Add()([downsampled_input1_twice_adjusted, downsampled_input2, input3, upsampled_input4_adjusted])

    # Adjust number of channels in downsampled input3, downsampled input2, and downsampled input1 to match input4
    downsampled_input3 = downsample_stride2_conv(input3, filters=96)
    downsampled_input2_twice = downsample_stride2_conv(downsample_stride2_conv(input2, filters=channels4), filters=channels4)
    downsampled_input2_twice_adjusted = tf.keras.layers.Conv2D(96, kernel_size=1, padding='same')(downsampled_input2_twice)
    downsampled_input1_four_times = downsample_stride2_conv(downsample_stride2_conv(downsample_stride2_conv(input1, filters=channels4), filters=channels4), filters=channels4)
    downsampled_input1_four_times_adjusted = tf.keras.layers.Conv2D(96, kernel_size=1, padding='same')(downsampled_input1_four_times)

    # Smallest output feature map
    output4 = tf.keras.layers.Add()([downsampled_input1_four_times_adjusted, downsampled_input2_twice_adjusted, downsampled_input3, input4])

    return output1, output2, output3, output4

In [8]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Input, Conv2D, Lambda
def rgb_stream(input_shape):
    # Load pre-trained ResNet50 model without the top classification layer
    #base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    
    # Freeze the layers of the base model to retain pre-trained weights
    #for layer in base_model.layers:
    #    layer.trainable = False

    inputs = Input(shape=input_shape)
    #x = base_model(inputs)  # Shape will be (None, 2, 2, 2048)
    
    # Upsample to match (64, 64)
    #x = Lambda(lambda image: tf.image.resize(image, (128, 128)))(x)  # Now shape is (None, 64, 64, 2048)
    
    # Stage 1
    x = convolutional_unit(inputs, filters1=32, filters2=32, num_repeats=1)
    x = convolutional_unit(x, filters1=128, filters2=128, num_repeats=1)
    
    # Stage 2 - Branching
    top_branch = convolutional_unit(x, filters1=24, filters2=24, num_repeats=1)
    middle_branch = downsample_stride2_conv(x, filters=48)
    
    top_branch = convolutional_unit(top_branch, filters1=24, filters2=24, num_repeats=1)
    middle_branch = convolutional_unit(middle_branch, filters1=48, filters2=48, num_repeats=1)
    
    # Stage 3
    top_temp = top_branch
    mid_temp = middle_branch

    third_branch = downsample_stride2_conv(downsample_stride2_conv(top_branch, filters=96), filters=96)
    third_branch = Add()([third_branch, downsample_stride2_conv(middle_branch, filters=96)])

    middle_branch_downsampled = downsample_stride2_conv(top_branch, filters=48)
    middle_branch = Conv2D(filters=48, kernel_size=1, padding='same')(middle_branch)
    middle_branch = Add()([middle_branch_downsampled, middle_branch])

    top_branch_upsampled = UpSample1x1Bilinear()(mid_temp)
    top_branch_upsampled = Conv2D(24, kernel_size=1, strides=1, padding='same')(top_branch_upsampled)
    top_branch = convolutional_unit(top_branch, filters1=24, filters2=24, num_repeats=1)
    top_branch = Add()([top_branch, top_branch_upsampled])

    # Repeating 4 times: Convolutional unit followed by fusion unit
    for _ in range(4):
        top_branch = convolutional_unit(top_branch, filters1=24, filters2=24, num_repeats=1)
        middle_branch = convolutional_unit(middle_branch, filters1=48, filters2=48, num_repeats=1)
        third_branch = convolutional_unit(third_branch, filters1=96, filters2=96, num_repeats=1)
        
        top_branch, middle_branch, third_branch = fusion_block_3inputs(top_branch, middle_branch, third_branch)
    
    # Adding the fourth parallel layer
    fourth_branch = downsample_stride2_conv(third_branch, filters=192)
    
    # Repeating 3 times: Convolutional unit followed by fusion unit
    for _ in range(3):
        top_branch = convolutional_unit(top_branch, filters1=24, filters2=24, num_repeats=1)
        middle_branch = convolutional_unit(middle_branch, filters1=48, filters2=48, num_repeats=1)
        third_branch = convolutional_unit(third_branch, filters1=96, filters2=96, num_repeats=1)
        fourth_branch = convolutional_unit(fourth_branch, filters1=192, filters2=192, num_repeats=1)
        
        top_branch, middle_branch, third_branch, fourth_branch = fusion_block_4inputs(top_branch, middle_branch, third_branch, fourth_branch)
    
    model = tf.keras.Model(inputs=inputs, outputs=[top_branch, middle_branch, third_branch, fourth_branch])
    return model

In [9]:
import tensorflow as tf

def fusion_block_2inputs(input1, input2):
    # Adjust input2 to have the same number of channels as input1
    input2_adjusted = tf.keras.layers.Conv2D(filters=input1.shape[-1], kernel_size=1, padding='same')(input2)

    # Upsample input2
    upsampled_input2 = UpSample1x1Bilinear(scale_factor=2)(input2_adjusted)

    # Output1: Element-wise addition
    output1 = tf.keras.layers.Add()([input1, upsampled_input2])

    # Downsample input1
    downsampled_input1 = downsample_stride2_conv(input1, filters=input2.shape[-1])

    # Output2: Element-wise addition
    output2 = tf.keras.layers.Add()([downsampled_input1, input2])

    return output1, output2

def dct_stream(input_shape):
    inputs = Input(shape=input_shape)
    quant_t = [[16, 11, 10, 16, 24, 40, 51, 61],
               [12, 12, 14, 19, 26, 58, 60, 55],
               [14, 13, 16, 24, 40, 57, 69, 56],
               [14, 17, 22, 29, 51, 87, 80, 62],
               [18, 22, 37, 56, 68, 109, 103, 77],
               [24, 35, 55, 64, 81, 104, 113, 92],
               [49, 64, 78, 87, 103, 121, 120, 101],
               [72, 92, 95, 98, 112, 100, 103, 99]]

    pretrained_jpegalm = JPEGALMLayer(quant_t, T=10, filters_dct=64, filters_quant=4)
    jpegalm_output = pretrained_jpegalm(inputs)
    # x = tf.keras.layers.Conv2D(filters=512, kernel_size=3, padding='same', activation='relu')(inputs)
    
    # Convolutional unit to produce a 96-channel map
    # x = convolutional_unit(x, filters1=96, filters2=96, num_repeats=1)

    x = Conv2D(filters=256, kernel_size=3, padding='same', activation='relu')(jpegalm_output)
    x = convolutional_unit(x, filters1=48, filters2=48, num_repeats=1)
    
    top_branch = convolutional_unit(x, filters1=48, filters2=48, num_repeats=1)
    middle_branch = downsample_stride2_conv(x, filters=96)

    for _ in range(3):
        top_branch = convolutional_unit(top_branch, filters1=48, filters2=48, num_repeats=1)
        middle_branch = convolutional_unit(middle_branch, filters1=96, filters2=96, num_repeats=1)
        top_branch, middle_branch = fusion_block_2inputs(top_branch, middle_branch)

    bottom_branch = downsample_stride2_conv(downsample_stride2_conv(top_branch, filters=192), filters=192)
    bottom_branch = Add()([bottom_branch, downsample_stride2_conv(middle_branch, filters=192)])

    for _ in range(2):
        top_branch = convolutional_unit(top_branch, filters1=48, filters2=48, num_repeats=1)
        middle_branch = convolutional_unit(middle_branch, filters1=96, filters2=96, num_repeats=1)
        bottom_branch = convolutional_unit(bottom_branch, filters1=192, filters2=192, num_repeats=1)
        top_branch, middle_branch, bottom_branch = fusion_block_3inputs(top_branch, middle_branch, bottom_branch)

    model = Model(inputs=inputs, outputs=[top_branch, middle_branch, bottom_branch])
    return model

In [10]:
import tensorflow as tf

class AdjustSpatialDimensions(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(AdjustSpatialDimensions, self).__init__(**kwargs)

    @tf.function
    def call(self, inputs):
        x, target_shape = inputs
        target_shape = tf.shape(target_shape)[1:3]
        resized = tf.image.resize(x, target_shape, method='bilinear')
        
        # Check for NaN values and raise an error if found
        tf.debugging.check_numerics(resized, "NaN values found in AdjustSpatialDimensions layer output")  
        return resized

def fusion_stream(rgb_model, dct_model):
    # Extracting outputs from the RGB and DCT streams
    rgb_outputs = rgb_model.outputs
    dct_outputs = dct_model.outputs

    print(dct_outputs[0])
    print(dct_outputs[1])
    print(dct_outputs[2])
    print(rgb_outputs[0])
    print(rgb_outputs[1])
    print(rgb_outputs[2])
    
    # Adjust spatial dimensions of DCT outputs to match RGB outputs
    adjust_layer = AdjustSpatialDimensions()
    dct_outputs[0] = adjust_layer([dct_outputs[0], rgb_outputs[1]])
    dct_outputs[1] = adjust_layer([dct_outputs[1], rgb_outputs[2]])
    dct_outputs[2] = adjust_layer([dct_outputs[2], rgb_outputs[3]])


    # Stage 1
    top_layer_rgb = convolutional_unit(rgb_outputs[0], filters1=24, filters2=6, num_repeats=1)

    # Stage 2
    combined_192 = tf.keras.layers.Concatenate()([rgb_outputs[1], dct_outputs[0]])  # 96 + 96 = 192 channels
    combined_192 = convolutional_unit(combined_192, filters1=96, filters2=24, num_repeats=1)

    # Stage 3
    combined_384 = tf.keras.layers.Concatenate()([rgb_outputs[2], dct_outputs[1]])  # 192 + 192 = 384 channels
    combined_384 = convolutional_unit(combined_384, filters1=192, filters2=48, num_repeats=1)

    # Stage 4
    combined_768 = tf.keras.layers.Concatenate()([rgb_outputs[3], dct_outputs[2]])  # 384 + 384 = 768 channels
    combined_768 = convolutional_unit(combined_768, filters1=384, filters2=96, num_repeats=1)

    # Apply fusion unit
    fused_output1, fused_output2, fused_output3, fused_output4 = fusion_block_4inputs_fusion_stream(top_layer_rgb, combined_192, combined_384, combined_768)

    # Final map of 360 channels
    fused_output1 = convolutional_unit(fused_output1, filters1=174, filters2=174, num_repeats=1)
    fused_output2 = UpSample1x1Bilinear(scale_factor=2)(fused_output2)
    fused_output3 = UpSample1x1Bilinear(scale_factor=4)(fused_output3)
    fused_output4 = UpSample1x1Bilinear(scale_factor=8)(fused_output4)

    final_combined = tf.keras.layers.Concatenate()([fused_output1, fused_output2, fused_output3, fused_output4])
    #final_output = convolutional_unit(final_combined, filters1=360, filters2=2, num_repeats=1)
    #final_output = Conv2D(2, (1, 1), activation='softmax')(final_combined)
    final_output = Conv2D(1, (1, 1), activation='sigmoid')(final_combined)

    model = tf.keras.Model(inputs=[rgb_model.input, dct_model.input], outputs=final_output)
    return model

In [11]:
def CATNet(input_shape_rgb, input_shape_dct):
    # Define the RGB and DCT models
    rgb_model = rgb_stream(input_shape_rgb)
    dct_model = dct_stream(input_shape_dct)

    # Create the fusion model
    model = fusion_stream(rgb_model, dct_model)
    
    return model

# Example usage
input_shape_rgb = (64, 64, 3)
input_shape_dct = (64, 64, 1)  # Assuming the input shape for DCT is single channel

# Create the CATNet model
catnet_model = CATNet(input_shape_rgb, input_shape_dct)

# Summarize the model
catnet_model.summary()

(None, 64, 64, 24)
(None, 64, 64, 24)
(None, 64, 64, 24)
(None, 64, 64, 24)
(None, 64, 64, 24)
(None, 64, 64, 24)
(None, 64, 64, 24)
(None, 64, 64, 24)
(None, 64, 64, 24)
(None, 64, 64, 24)
(None, 64, 64, 24)
(None, 64, 64, 24)
(None, 32, 32, 48)
(None, 32, 32, 48)
(None, 32, 32, 48)
(None, 32, 32, 48)
(None, 32, 32, 48)
(None, 32, 32, 48)
(None, 32, 32, 48)
(None, 32, 32, 48)
(None, 32, 32, 48)
(None, 32, 32, 48)
(None, 32, 32, 48)
(None, 32, 32, 48)
(None, 64, 64, 48)
(None, 64, 64, 48)
(None, 64, 64, 48)
(None, 64, 64, 48)
(None, 64, 64, 48)
(None, 64, 64, 48)
KerasTensor(type_spec=TensorSpec(shape=(None, 64, 64, 48), dtype=tf.float32, name=None), name='add_81/add_1:0', description="created by layer 'add_81'")
KerasTensor(type_spec=TensorSpec(shape=(None, 32, 32, 96), dtype=tf.float32, name=None), name='add_82/add_1:0', description="created by layer 'add_82'")
KerasTensor(type_spec=TensorSpec(shape=(None, 16, 16, 192), dtype=tf.float32, name=None), name='add_83/add_1:0', description

In [12]:
import os
import numpy as np
import tensorflow as tf
import cv2
import jpegio as jio
import gc
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# Clear memory before training
gc.collect()

# Define paths
base_dir = r'C:\fakeimage\archive\CASIA2'
au_dir = os.path.join(base_dir, 'Au')
sp_dir = os.path.join(base_dir, 'Tp')
gt_dir = os.path.join(base_dir, 'groundtruth')

# Parameters
img_height = 64
img_width = 64
batch_size = 1  # Adjust batch size

# Collect file paths and labels
au_filepaths = [os.path.join(au_dir, fname) for fname in os.listdir(au_dir) if fname.endswith('.jpg') or fname.endswith('.tif')]
sp_filepaths = [os.path.join(sp_dir, fname) for fname in os.listdir(sp_dir) if fname.endswith('.jpg') or fname.endswith('.tif')]
filepaths = au_filepaths + sp_filepaths
labels = [0] * len(au_filepaths) + [1] * len(sp_filepaths)

# Shuffle file paths and labels together
data = list(zip(filepaths, labels))
np.random.shuffle(data)
filepaths, labels = zip(*data)

# Split into training and validation
split_idx = int(len(filepaths) * 0.8)
train_filepaths, val_filepaths = filepaths[:split_idx], filepaths[split_idx:]
train_labels, val_labels = labels[:split_idx], labels[split_idx:]

# Helper function to load and preprocess image
def load_image(filepath, img_height, img_width):
    img = load_img(filepath, target_size=(img_height, img_width))
    img = img_to_array(img) / 255.0
    return img

# Helper function to extract DCT coefficients
def load_dct(filepath, img_height, img_width):
    jpeg_struct = jio.read(filepath)
    dct_coeff = jpeg_struct.coef_arrays[0]  # Assuming Y channel
    dct_coeff = dct_coeff.astype(np.float32)
    dct_coeff = dct_coeff / np.max(np.abs(dct_coeff) + 1e-8)  # Avoid division by zero
    dct_coeff = cv2.resize(dct_coeff, (img_width, img_height))
    dct_coeff = dct_coeff[:, :, np.newaxis]  # Add channel dimension
    return dct_coeff

# Helper function to load and preprocess ground truth mask
def load_mask(filepath, img_height, img_width):
    mask = load_img(filepath, color_mode="grayscale", target_size=(img_height, img_width))
    mask = img_to_array(mask) / 255.0  # Normalize to [0, 1]
    mask = mask.astype(np.int32)  # Convert to integer type
    return mask

# Create a TensorFlow Dataset
def create_dataset(filepaths, labels, img_height, img_width, batch_size, repeat=True):
    def generator():
        for filepath, label in zip(filepaths, labels):
            img = load_image(filepath, img_height, img_width)
            dct = load_dct(filepath, img_height, img_width)
            if label == 1:  # Tampered
                mask_filepath = os.path.join(gt_dir, os.path.basename(filepath).split('.')[0] +'_gt'+ '.png')
                label_img = load_mask(mask_filepath, img_height, img_width)
            else:  # Authentic
                label_img = np.zeros((img_height, img_width, 1), dtype=np.int32)
            if np.isnan(img).any() or np.isnan(dct).any():
                print(f"NaN found in input data for file: {filepath}")
            yield (img, dct), label_img

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            (tf.TensorSpec(shape=(img_height, img_width, 3), dtype=tf.float32),
             tf.TensorSpec(shape=(img_height, img_width, 1), dtype=tf.float32)),
            tf.TensorSpec(shape=(img_height, img_width, 1), dtype=tf.int32)
        )
    )
    if repeat:
        dataset = dataset.repeat()  # Repeat the dataset indefinitely
    dataset = dataset.shuffle(buffer_size=len(filepaths)).batch(batch_size).prefetch(tf.data.experimental.AUTOTUNE)
    return dataset

# Create datasets
train_dataset = create_dataset(train_filepaths, train_labels, img_height, img_width, batch_size)
val_dataset = create_dataset(val_filepaths, val_labels, img_height, img_width, batch_size)


In [13]:
import tensorflow as tf
tf.keras.backend.clear_session()

In [14]:
"""
import os
import numpy as np
import tensorflow as tf
import cv2
import jpegio as jio
import gc
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Flatten, Dense, Concatenate, Activation
from tensorflow.keras.callbacks import ModelCheckpoint, LearningRateScheduler
from tensorflow.keras import mixed_precision
# Clear memory before training
gc.collect()

# Enable mixed precision
#tf.keras.mixed_precision.set_global_policy('mixed_float16')

# Define paths
base_dir = r'C:\fakeimage\archive\CASIA2'
au_dir = os.path.join(base_dir, 'Au')
sp_dir = os.path.join(base_dir, 'Tp')

# Parameters
img_height = 256
img_width = 256
batch_size = 2  # Reduced batch size to reduce memory usage

# Define the model with corrected input shapes
input_rgb = Input(shape=(img_height, img_width, 3))
input_dct = Input(shape=(img_height, img_width, 1))

# Process RGB input
#x_rgb = Conv2D(32, (3, 3), activation='relu')(input_rgb)
#x_rgb = Flatten()(x_rgb)

# Process DCT input
#x_dct = Conv2D(32, (3, 3), activation='relu')(input_dct)
#x_dct = Flatten()(x_dct)

# Concatenate features
#x = Concatenate()([x_rgb, x_dct])
#x = Dense(64, activation='relu')(x)

# Correct the number of neurons in the output layer to 2
#x = Dense(2, name='dense_logits')(x)
#outputs = Activation('softmax', dtype='float32', name='predictions')(x)

# Define model
#catnet_model = Model(inputs=[input_rgb, input_dct], outputs=outputs)

# Compile the model
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9, nesterov=True, clipvalue=1.0)
catnet_model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Learning rate scheduler
def scheduler(epoch, lr):
    if epoch % 10 == 0 and epoch != 0:
        lr = lr * 0.1
    return lr

callback_lr = LearningRateScheduler(scheduler)

# Custom callback to save the best model based on multiple metrics
class SaveBestModel(tf.keras.callbacks.Callback):
    def __init__(self, filepath, monitor_metrics={'val_loss': 'min', 'val_accuracy': 'max'}):
        super(SaveBestModel, self).__init__()
        self.filepath = filepath
        self.monitor_metrics = monitor_metrics
        self.best_metrics = {metric: np.inf if mode == 'min' else -np.inf for metric, mode in monitor_metrics.items()}

    def on_epoch_end(self, epoch, logs=None):
        save = False
        for metric, mode in self.monitor_metrics.items():
            current_metric_value = logs.get(metric)
            if current_metric_value is not None:
                if (mode == 'min' and current_metric_value < self.best_metrics[metric]) or \
                   (mode == 'max' and current_metric_value > self.best_metrics[metric]):
                    self.best_metrics[metric] = current_metric_value
                    save = True
        if save:
            self.model.save(self.filepath.format(epoch=epoch, **logs))
            print(f"Saved model at epoch {epoch} based on improved metrics: {self.best_metrics}")

# Model checkpoint callback to save only weights
checkpoint_filepath = '256x256casia1_simplecatnet.weights.keras'
checkpoint_callback = SaveBestModel(
    filepath=checkpoint_filepath,
    monitor_metrics={'val_loss': 'min', 'val_accuracy': 'max'}
)

# Train the model
history = catnet_model.fit(
    train_dataset,
    steps_per_epoch=len(train_filepaths) // batch_size,
    epochs=200,
    validation_data=val_dataset,
    validation_steps=len(val_filepaths) // batch_size,
    callbacks=[callback_lr, checkpoint_callback]
)

# Load the best model weights
catnet_model.load_weights(checkpoint_filepath)
"""

'\nimport os\nimport numpy as np\nimport tensorflow as tf\nimport cv2\nimport jpegio as jio\nimport gc\nfrom tensorflow.keras.models import Model\nfrom tensorflow.keras.layers import Input, Conv2D, Flatten, Dense, Concatenate, Activation\nfrom tensorflow.keras.callbacks import ModelCheckpoint, LearningRateScheduler\nfrom tensorflow.keras import mixed_precision\n# Clear memory before training\ngc.collect()\n\n# Enable mixed precision\n#tf.keras.mixed_precision.set_global_policy(\'mixed_float16\')\n\n# Define paths\nbase_dir = r\'C:\x0cakeimage\x07rchive\\CASIA2\'\nau_dir = os.path.join(base_dir, \'Au\')\nsp_dir = os.path.join(base_dir, \'Tp\')\n\n# Parameters\nimg_height = 256\nimg_width = 256\nbatch_size = 2  # Reduced batch size to reduce memory usage\n\n# Define the model with corrected input shapes\ninput_rgb = Input(shape=(img_height, img_width, 3))\ninput_dct = Input(shape=(img_height, img_width, 1))\n\n# Process RGB input\n#x_rgb = Conv2D(32, (3, 3), activation=\'relu\')(input_rg

In [ ]:
import os
import numpy as np
import tensorflow as tf
import cv2
import jpegio as jio
import gc
import PIL.Image
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Flatten, Dense, Concatenate, Activation
from tensorflow.keras.callbacks import ModelCheckpoint, LearningRateScheduler
from tensorflow.keras import mixed_precision

# Clear memory before training
gc.collect()

# Enable mixed precision
#tf.keras.mixed_precision.set_global_policy('mixed_float16')

# Define paths
#base_dir = r'C:\fakeimage\archive\CASIA2'
#au_dir = os.path.join(base_dir, 'Au')
#sp_dir = os.path.join(base_dir, 'Tp')

# Parameters
#img_height = 256
#img_width = 256
#batch_size = 8  # Reduced batch size to reduce memory usage

# Define the model with corrected input shapes
#input_rgb = Input(shape=(img_height, img_width, 3))
#input_dct = Input(shape=(img_height, img_width, 1))

# Process RGB input
#x_rgb = Conv2D(32, (3, 3), activation='relu')(input_rgb)
#x_rgb = Flatten()(x_rgb)

# Process DCT input
#x_dct = Conv2D(32, (3, 3), activation='relu')(input_dct)
#x_dct = Flatten()(x_dct)

# Concatenate features
#x = Concatenate()([x_rgb, x_dct])
#x = Dense(64, activation='relu')(x)

# Correct the number of neurons in the output layer to 2
#x = Dense(2, name='dense_logits')(x)
#outputs = Activation('softmax', dtype='float32', name='predictions')(x)

# Define model
#catnet_model = Model(inputs=[input_rgb, input_dct], outputs=outputs)

# Compile the model
optimizer = tf.keras.optimizers.SGD(learning_rate=0.0005, momentum=0.9, nesterov=True, clipvalue=1.0)
catnet_model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Learning rate scheduler
def scheduler(epoch, lr):
    if epoch % 10 == 0 and epoch != 0:
        lr = lr * 0.1
    return lr

callback_lr = LearningRateScheduler(scheduler)

import tensorflow as tf
from tensorflow.keras.callbacks import Callback

class SaveBestModel(Callback):
    def __init__(self, filepath, monitor_metrics):
        super(SaveBestModel, self).__init__()
        self.filepath = filepath
        self.monitor_metrics = monitor_metrics
        self.best_metrics = {key: None for key in monitor_metrics}

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        save = False
        for key, mode in self.monitor_metrics.items():
            current = logs.get(key)
            if current is not None:
                if self.best_metrics[key] is None or \
                        (mode == 'min' and current < self.best_metrics[key]) or \
                        (mode == 'max' and current > self.best_metrics[key]):
                    self.best_metrics[key] = current
                    save = True
        if save:
            save_path = self.filepath.format(epoch=epoch, **logs)
            tf.saved_model.save(self.model, save_path)
            print(f"Saved model at epoch {epoch} based on improved metrics: {self.best_metrics}")

# Assuming you have defined `catnet_model`, `train_dataset`, `val_dataset`, `train_filepaths`, `val_filepaths`, and `batch_size`

checkpoint_filepath = r'/savedmodels/saved_model_epoch_{epoch:02d}_val_loss_{val_loss:.2f}.pb'
checkpoint_callback = SaveBestModel(
    filepath=checkpoint_filepath,
    monitor_metrics={'val_loss': 'min', 'val_accuracy': 'max'}
)

history = catnet_model.fit(
    train_dataset,
    steps_per_epoch=len(train_filepaths) // batch_size,
    epochs=20,
    validation_data=val_dataset,
    validation_steps=len(val_filepaths) // batch_size,
    callbacks=[checkpoint_callback]
)


Epoch 1/20


In [ ]:
# Save the model
catnet_model.save('128x128casia2_simplecatnet.keras')

In [ ]:
catnet_model.save('128x128casia2_simplecatnet.h5')

In [ ]:
import matplotlib.pyplot as plt
# Plotting the training and validation loss and accuracy
def plot_training_history(history):
    # Extract the history dictionary
    history_dict = history.history
    
    # Plot training & validation accuracy values
    plt.figure(figsize=(12, 5))
    
    # Plot accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history_dict['accuracy'], label='Training Accuracy')
    plt.plot(history_dict['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend(loc='upper left')
    
    # Plot loss
    plt.subplot(1, 2, 2)
    plt.plot(history_dict['loss'], label='Training Loss')
    plt.plot(history_dict['val_loss'], label='Validation Loss')
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend(loc='upper left')
    
    plt.tight_layout()
    plt.show()

# Call the function to plot the training history
plot_training_history(history)

In [ ]:
from tensorflow.keras.models import load_model

# Load the entire model
best_catnet_model = catnet_model.load_weights('best_model_.weights.h5')

# Now you can use best_catnet_model for inference or further training
catnet_model.save('256_384_3rdmodel.keras')

In [ ]:
import tensorflow as tf
import jpegio as jio
import numpy as np
import cv2
import matplotlib.pyplot as plt
# Set GPU memory growth
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        print(e)  # Memory growth must be set before GPUs have been initialized

# Define the preprocess_image function
def preprocess_image(image_path, target_size=(128, 128)):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=target_size)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = img_array / 255.0  # Rescale
    return img_array

# Define the preprocess_dct function
def preprocess_dct(image_path, target_size=(128, 128)):
    jpeg_struct = jio.read(image_path)
    dct_coeff = jpeg_struct.coef_arrays[0]  # Assuming Y channel
    dct_coeff = dct_coeff / np.max(np.abs(dct_coeff))
    dct_coeff = cv2.resize(dct_coeff, target_size)
    dct_coeff = dct_coeff[:, :, np.newaxis]  # Add channel dimension
    return dct_coeff
    

In [ ]:
# Load the best model
loaded_model = tf.keras.models.load_model('16x16_casia1_simplecatnet.keras')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Path to the image to be predicted
image_path = r'C:\fakeimage\archive\CASIA\Tp\Tp_D_NRD_S_N_cha10107_cha10106_11119.jpg'

# Preprocess the image and DCT coefficients
preprocessed_image = preprocess_image(image_path)
preprocessed_dct = preprocess_dct(image_path)

# Expand dimensions to match the batch size expected by the model
preprocessed_image = np.expand_dims(preprocessed_image, axis=0)
preprocessed_dct = np.expand_dims(preprocessed_dct, axis=0)

# Make predictions
prediction = catnet_model.predict([preprocessed_image, preprocessed_dct])

# Print the shape of the prediction output
print("Prediction shape:", prediction.shape)

# Extract the first sample's prediction map
probability_map = prediction[0]

# Assuming the second channel is the class of interest for visualization
manipulated_probability_map = probability_map[:, :, 1]

def display_image_with_prediction(image_path, probability_map):
    # Load the original image
    original_image = tf.keras.preprocessing.image.load_img(image_path)
    original_image = tf.keras.preprocessing.image.img_to_array(original_image)
    original_image = original_image / 255.0  # Rescale

    # Display the images and probability map
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(original_image)

    plt.subplot(1, 2, 2)
    plt.title("Manipulated Probability Map")
    plt.imshow(probability_map, cmap='viridis')
    plt.colorbar()
    plt.xlabel('Pixel X')
    plt.ylabel('Pixel Y')

    plt.show()

# Display the original image and the probability map
display_image_with_prediction(image_path, manipulated_probability_map)


In [ ]:
# Display the image with prediction
display_image_with_prediction(image_path, probability_vector)

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
import jpegio as jio

# Define the preprocess_image function
def preprocess_image(image_path, target_size=(128, 128)):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=target_size)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = img_array / 255.0  # Rescale
    return img_array

# Define the preprocess_dct function
def preprocess_dct(image_path, target_size=(128, 128)):
    jpeg_struct = jio.read(image_path)
    dct_coeff = jpeg_struct.coef_arrays[0]  # Assuming Y channel
    dct_coeff = dct_coeff / np.max(np.abs(dct_coeff))
    dct_coeff = cv2.resize(dct_coeff, target_size)
    dct_coeff = dct_coeff[:, :, np.newaxis]  # Add channel dimension
    return dct_coeff

# Load the best model on CPU
with tf.device('/CPU:0'):
    loaded_model = tf.keras.models.load_model('256x256casia2_simplecatnet.keras')

# Path to the image to be predicted
image_path = r'C:\fakeimage\archive\CASIA2\Tp\Tp_D_CND_S_N_txt00028_txt00006_10848.jpg'

# Preprocess the image and DCT coefficients
preprocessed_image = preprocess_image(image_path)
preprocessed_dct = preprocess_dct(image_path)

# Expand dimensions to match the batch size expected by the model
preprocessed_image = np.expand_dims(preprocessed_image, axis=0)
preprocessed_dct = np.expand_dims(preprocessed_dct, axis=0)

# Make predictions
prediction = loaded_model.predict([preprocessed_image, preprocessed_dct])

# Print the shape of the prediction output
print("Prediction shape:", prediction.shape)

# Check if prediction is 1D and reshape if necessary (assuming the model's output is intended to be a 2D map)
if prediction.ndim == 2 and prediction.shape[1] == 10:
    # Assuming the model output is actually a flat array of a 2D map, reshape it (for example to 64x64)
    side_length = int(np.sqrt(prediction.shape[1]))
    probability_map = prediction.reshape((side_length, side_length))
else:
    # If prediction is not 1D or reshaping does not make sense, raise an error or handle appropriately
    raise ValueError(f"Unexpected prediction shape: {prediction.shape}")

# Normalize the prediction to 0-1
probability_map = (probability_map - np.min(probability_map)) / (np.max(probability_map) - np.min(probability_map))

# Apply a threshold to get a binary tampering map
threshold = 0.5
tampering_map = (probability_map > threshold).astype(np.uint8) * 255

# Display the original image and the tampering map
def display_image_with_prediction(image_path, tampering_map):
    # Load the original image
    original_image = tf.keras.preprocessing.image.load_img(image_path)
    original_image = tf.keras.preprocessing.image.img_to_array(original_image)
    original_image = original_image / 255.0  # Rescale

    # Display the images
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(original_image)

    plt.subplot(1, 2, 2)
    plt.title("Tampering Map")
    plt.imshow(tampering_map, cmap='gray')  # Display in grayscale

    plt.show()

# Display the image with prediction
display_image_with_prediction(image_path, tampering_map)